##  Step 2: **Configure API Keys & Imports**

In [ ]:
# Fix numpy binary incompatibility (dtype size mismatch)
import subprocess
subprocess.run(["pip", "install", "-q", "--force-reinstall", "numpy==1.26.4"], check=True)

# Must restart the runtime after reinstalling numpy
import os
print("\u2705 numpy reinstalled. Restarting runtime...")
os.kill(os.getpid(), 9)

In [2]:
import os, re, warnings
from pathlib import Path
from typing import List, Tuple, Optional
warnings.filterwarnings("ignore")

# ── Compatibility shim: patch missing HfFolder before Gradio imports it ───────
# Colab's pre-installed huggingface_hub >= 0.24 removed HfFolder.
# Gradio 4.x still references it, so we inject a minimal stub.
try:
    from huggingface_hub import HfFolder  # works if version is already compatible
except ImportError:
    import huggingface_hub as _hfhub
    class _HfFolder:
        @staticmethod
        def get_token(): return None
        @staticmethod
        def save_token(token): pass
    _hfhub.HfFolder = _HfFolder
    import sys
    sys.modules["huggingface_hub"].HfFolder = _HfFolder
    print("🔧 huggingface_hub.HfFolder shim applied.")

# ── Now Gradio imports cleanly ────────────────────────────────────────────────
import gradio as gr
print("\n\u2705 Gradio imported successfully.")

# ── Load Gemini API key from Colab Secrets ────────────────────────────────────
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    if not GEMINI_API_KEY:
        raise ValueError("Secret is empty")
    print("\n\u2705 Gemini API key loaded from Colab Secrets.")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
    if GEMINI_API_KEY:
        print("\n\u2705 Gemini API key loaded from environment variable.")
    else:
        print("WARNING: GEMINI_API_KEY not found!")
        print("   Add it to Colab Secrets (icon in left sidebar).")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# ── Core framework imports ────────────────────────────────────────────────────
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyMuPDFLoader, Docx2txtLoader

import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)
print("\n\u2705 All imports loaded successfully!")

🔧 huggingface_hub.HfFolder shim applied.

✅ Gradio imported successfully.

✅ Gemini API key loaded from Colab Secrets.

✅ All imports loaded successfully!
